<a href="https://colab.research.google.com/github/LavenaD/Medical-Test-Summarisation/blob/main/TrainModel_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Install the libraries required to import the model from Hugging face**

In [2]:
!pip install torch transformers datasets accelerate rouge-score evaluate pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0db01a4d216a4ad5467e8926676c6df07b10213510cc520839b1db21eeb0ed0a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


**2. Read the data after the data is cleaned and stored in a csv file**

In [3]:
import pandas as pd
df = pd.read_csv('/content/sample_data/output_20260424114330_Train_Full.csv')
df.head()

,id,findings,labels
0,3997,"Heart size within normal limits. Small, nodula...","No acute findings, no evidence for active TB."
1,3996,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...
2,3995,The cardiomediastinal silhouette and pulmonary...,1. Interval resolution of bibasilar airspace d...
3,3994,Similar mild cardiomegaly. Of the pulmonary va...,Mild cardiomegaly with XXXX of early failure.
4,3993,The heart is mildly enlarged. Left hemidiaphra...,Borderline cardiomegaly without acute disease.


**3. Include additional text in the findings column which will summarize the text.**

In [4]:
df["findings"] = "Summarize this medical condition: " + df["findings"]

In [5]:
df.head()
print(df['findings'][0])
print(len(df['findings'][0]))

Summarize this medical condition: Heart size within normal limits. Small, nodular opacity in the right upper lobe. This does not look like an acute infiltrate, and more XXXX represents a granuloma. No pneumothorax or effusions.
227


**4. Download the model from huggingface**

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

**5. Use Lora to train the model**

In [7]:
import torchao
print(f"Installed torchao version: {torchao.__version__}")

Installed torchao version: 0.10.0


In [8]:
!pip uninstall torchao -y

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [9]:
import sys
!{sys.executable} -m pip install torchao>=0.16.0
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=14,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)

**6. Set the training arguments**

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_model",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=4,
    learning_rate=8.8e-5,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch"
)

**7. Split the data into train and test**

In [11]:
from transformers import Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split the dataset into train and test sets using the Dataset's own split method
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_datasets["train"]
test_dataset = split_datasets["test"]



**8. Tokenize the data**

In [12]:
# Define the tokenization function
def tokenize_function(examples):
  # return tokenizer(examples["findings"],
  #                  text_target=examples["labels"],
  #                  truncation=True)
  return tokenizer(
        [str(f) for f in examples["findings"]],
        text_target=[str(l) for l in examples["labels"]],
        truncation=True
    )

# Apply tokenization to the datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove the original text columns as they are no longer needed
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["id", "findings"])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["id", "findings"])

# Make sure the 'labels' column is correctly set for training
tokenized_train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset
)

trainer.train()

Map:   0%|          | 0/1525 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,8.640622,8.328773
2,8.473970,8.084224
3,8.390509,8.012403
4,8.407912,7.991953


TrainOutput(global_step=6100, training_loss=8.575549486504226, metrics={'train_runtime': 674.1469, 'train_samples_per_second': 9.048, 'train_steps_per_second': 9.048, 'total_flos': 153159329292288.0, 'train_loss': 8.575549486504226, 'epoch': 4.0})

**9. Save the model**

In [13]:
import os
output_dir = "/content/medical_model/compressed-artifacts-g5/"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained("/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft")
tokenizer.save_pretrained("/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft")

('/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft/tokenizer_config.json',
 '/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft/tokenizer.json')

In [14]:
from transformers import AutoModelForSeq2SeqLM

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
from peft import PeftModel

print(type(model))

<class 'peft.peft_model.PeftModelForSeq2SeqLM'>


In [16]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft"
)

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft"
)

**10. Run Predictions**

In [18]:
import pandas as pd
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

model_path = "/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft"

# load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# attach LoRA adapter
model = PeftModel.from_pretrained(base_model, model_path)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

device = torch.device("cpu")
model.to(device)

# load data
test_df = pd.read_csv("/content/sample_data/output_20260424115121_validation.csv")

inputs = test_df["findings"].tolist()
references = test_df["labels"].tolist()

# predict
predictions = []

# inputs = inputs[0:10]

for text in inputs:

    text = ''' Summarize this medical condition: ''' + text

    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    output = model.generate(
        **tokens,
        max_length=256,
        num_beams=5
    )

    pred = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    predictions.append(pred)

print(predictions)
print(inputs)
print(references)



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary abnormality

In [19]:
!pip install evaluate rouge_score

In [20]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=predictions,
    references=references
)

print(results)

{'rouge1': np.float64(0.4628855976564419), 'rouge2': np.float64(0.3112153235106317), 'rougeL': np.float64(0.4619403533062294), 'rougeLsum': np.float64(0.4624617903033378)}


In [21]:
print(predictions)

['No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary disease.', 'No acute cardiopulmonary abnormality

In [22]:
import zipfile
import os
from google.colab import files

folder_to_compress = "/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft"
output_zip_name = "medical_summarizer_model_g5.zip"

# Create a ZipFile object
with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_folder in os.walk(folder_to_compress):
        for file in files_in_folder:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to folder_to_compress
            zipf.write(file_path, os.path.relpath(file_path, folder_to_compress))

print(f"'{folder_to_compress}' successfully compressed to '{output_zip_name}'")

# Download the zip file
files.download(output_zip_name)

'/content/medical_model/compressed-artifacts-g5/medical_summarizer_peft' successfully compressed to 'medical_summarizer_model_g5.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>